In [ ]:
# Setup cell. Canonical text: paste this file into the first code cell of every stage notebook.
STAGE = 5  # the only line that differs between notebooks

import contextlib
import json
import os
import shutil
import subprocess
import sys
import tarfile
import urllib.request
from pathlib import Path

ON_COLAB = "google.colab" in sys.modules
OBJECT = "my-object"  # folder name under real-to-sim-twin/ for your capture
RECORDING_ID = "real-to-sim-twin"
REFERENCE_URL = "https://artifacts.aegeanai.com/projects/real-to-sim-twin/reference"

RERUN = "rerun-sdk[notebook]==0.41.0"
PACKAGES = {
    1: ["pycolmap==4.2.1"],
    2: ["git+https://github.com/facebookresearch/sam2@2b90b9f", "huggingface_hub", "gsplat==1.5.3", "ninja",
        "git+https://github.com/rmbrualla/pycolmap@cc7ea4b"],
    3: ["gsplat==1.5.3", "ninja", "open3d==0.20.0", "git+https://github.com/rmbrualla/pycolmap@cc7ea4b"],
    4: ["diffusers==0.42.1", "transformers==5.21.0", "xatlas==0.0.11", "open3d==0.20.0", "pycolmap==4.2.1"],
    "4b": ["lpips==0.1.4", "pycolmap==4.2.1", "ninja"],
    5: ["trimesh==5.2.0", "coacd==1.0.14"],
    6: ["newton[sim,importers]==1.7.0", "usd-core==26.8", "trimesh==5.2.0", "lpips==0.1.4", "gsplat==1.5.3", "ninja",
        "pycolmap==4.2.1", "open3d==0.20.0"],
}
# Built against the installed torch, so pip must not isolate the build.
NO_BUILD_ISOLATION = {"4b": ["git+https://github.com/NVlabs/nvdiffrast@v0.4.0"],
                      6: ["git+https://github.com/NVlabs/nvdiffrast@v0.4.0"]}
# Reference objects with a course-made TRELLIS.2 archive, generated-<object>.tar.gz.
GENERATED_OBJECTS = ("boxed_beverage_001", "hammer_014")
# The TRELLIS.2 model revision those archives were generated with (model_revision in trellis2.json).
GENERATED_MODEL_REVISION = "af44b45f2e35a493886929c6d786e563ec68364d"


def _pip(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)


def _patch_pycolmap_fork():
    # The fork breaks on numpy 2: np.uint64(-1) raises OverflowError.
    import importlib
    import importlib.util
    importlib.invalidate_caches()  # the package was installed by pip in this process
    root = Path(importlib.util.find_spec("pycolmap").submodule_search_locations[0])
    for f in root.rglob("*.py"):
        text = f.read_text()
        if "np.uint64(-1)" in text:
            f.write_text(text.replace("np.uint64(-1)", "np.uint64(2**64-1)"))


if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    _pip(RERUN, *PACKAGES[STAGE])
    if STAGE in NO_BUILD_ISOLATION:
        _pip("--no-build-isolation", *NO_BUILD_ISOLATION[STAGE])
    if STAGE in (2, 3):  # stages 2 and 3 read stage1/sparse/0 with the pycolmap fork, never the official wheel
        _patch_pycolmap_fork()
    PROJECT = Path(f"/content/drive/MyDrive/real-to-sim-twin/{OBJECT}")
else:
    if "R2S_PROJECT" not in os.environ:
        raise RuntimeError("Off Colab, set the R2S_PROJECT environment variable to the project folder.")
    PROJECT = Path(os.environ["R2S_PROJECT"])
PROJECT.mkdir(parents=True, exist_ok=True)


@contextlib.contextmanager
def atomic_dir(path):
    """Yield path.tmp; on success it replaces path, on failure path is untouched."""
    path = Path(path)
    tmp, old = path.with_name(path.name + ".tmp"), path.with_name(path.name + ".old")
    shutil.rmtree(tmp, ignore_errors=True)
    if old.exists() and not path.exists():
        old.rename(path)  # a crash between the two renames left the only good copy in .old
    shutil.rmtree(old, ignore_errors=True)
    tmp.mkdir(parents=True)
    try:
        yield tmp
        if path.exists():
            path.rename(old)
        tmp.rename(path)
    except BaseException:
        shutil.rmtree(tmp, ignore_errors=True)
        if old.exists() and not path.exists():
            old.rename(path)
        raise
    shutil.rmtree(old, ignore_errors=True)


def prev_folder(stage):
    """The folder a stage reads: capture/ for stage 1, stage4/ for stage 4b, stage<N-1>/ otherwise.

    Stage 5 reads stage4/ too: stage 4b refines the maps but leaves the mesh to stage 4.
    """
    if stage == 1:
        return "capture"
    if stage == "4b":
        return "stage4"
    return f"stage{stage - 1}"


def on_reference(stage):
    """True if this stage runs on the reference object.

    The reference archives mark each stage folder with an empty .reference file, and a stage
    run on the reference marks the folder it writes. So the answer comes from the stage's own
    input folder (prev_folder(stage)), not from ground_truth/, which
    stays in the project after any fallback. For stage 1 a marked capture/ that also holds files
    of your own (own_capture_files) is not the reference.
    """
    if not (PROJECT / prev_folder(stage) / ".reference").is_file():
        return False
    return not (stage == 1 and own_capture_files(PROJECT / "capture"))


VIDEO_SUFFIXES = (".mp4", ".mov", ".m4v", ".avi", ".mkv", ".webm")


def own_capture_files(capture) -> list:
    """Files in capture/ that the reference capture never holds, so they must be yours.

    The reference capture has no board.json, and its only video is video.mp4, extracted at the same
    time as its .reference marker. So board.json, a video under another name (an iPhone's .mov, say)
    or a video changed more than a minute after the marker means that you added your own capture to
    a folder where a reference run left its files. board.json is the signal that does not depend on
    file times; your own capture always needs it for the metric scale.
    """
    capture = Path(capture)
    marker = capture / ".reference"
    since = marker.stat().st_mtime if marker.exists() else float("-inf")
    own = ["board.json"] if (capture / "board.json").exists() else []
    for p in sorted(capture.glob("*")):
        if p.suffix.lower() in VIDEO_SUFFIXES and (p.name != "video.mp4" or p.stat().st_mtime > since + 60):
            own.append(p.name)
    return own


def capture_video(capture):
    """The capture's video: video.mp4, or the one video file of another common phone format."""
    capture = Path(capture)
    if (capture / "video.mp4").exists():
        return capture / "video.mp4"
    videos = [p for p in sorted(capture.glob("*")) if p.suffix.lower() in VIDEO_SUFFIXES]
    if len(videos) != 1:
        raise FileNotFoundError(f"capture/ needs exactly one video (video.mp4, or one .mov, .m4v, .avi, .mkv or "
                                f".webm file); it holds {[p.name for p in videos] or 'none'}")
    return videos[0]


def _extract_reference(archive, names=None, dest=None):
    """Download a reference archive and move its top-level folders (or only `names`) into dest.

    dest defaults to PROJECT. A non-empty folder of yours that a reference folder replaces is
    kept as <name>.yours (an older .yours is replaced); ground_truth/ is replaced without a copy.
    Nothing moves until the archive is fully extracted.
    """
    dest = PROJECT if dest is None else Path(dest)
    base = os.environ.get("R2S_REFERENCE_URL", REFERENCE_URL).rstrip("/")
    tmp = dest / f".{archive}.part"
    req = urllib.request.Request(f"{base}/{archive}", headers={"User-Agent": "real-to-sim-twin"})
    with urllib.request.urlopen(req) as r, open(tmp, "wb") as f:
        shutil.copyfileobj(r, f)
    scratch = dest / ".extract"
    shutil.rmtree(scratch, ignore_errors=True)
    try:
        with tarfile.open(tmp) as tar:
            tar.extractall(scratch, filter="data")
        for item in scratch.iterdir():  # move into place only after a complete extraction
            if names is not None and item.name not in names:
                continue
            target = dest / item.name
            if target.is_dir() and any(target.iterdir()) and item.name != "ground_truth":
                yours = dest / f"{item.name}.yours"
                shutil.rmtree(yours, ignore_errors=True)
                target.rename(yours)
                print(f"your {item.name}/ is kept as {yours.name}/ and replaced by the reference {item.name}/.")
            elif target.is_dir():
                shutil.rmtree(target)
            elif target.exists():
                target.unlink()
            item.rename(target)
    finally:
        shutil.rmtree(scratch, ignore_errors=True)
        tmp.unlink(missing_ok=True)


def ensure_reference_capture(stage):
    """Stages 5 and 6 on the reference object use the reference capture/ too. True if it fetched it.

    When the stage's input folder is the reference object's (on_reference(stage), which follows
    the .reference markers from any earlier fallback) but capture/ is not (no capture/.reference),
    your measurements in capture/ belong to a different object than the mesh. The reference
    capture/ (physical.json, and the second-light files for stage 6) is then taken from the
    previous stage's archive, and yours is kept as capture.yours/.
    """
    if stage not in (5, 6) or not on_reference(stage) or (PROJECT / "capture" / ".reference").is_file():
        return False
    archive = f"{prev_folder(stage)}.tar.gz"
    print(f"{prev_folder(stage)}/ holds the reference object, so this stage uses the reference capture/ ({archive}).")
    _extract_reference(archive, names={"capture"})
    return True


def ensure_inputs(stage):
    """True if it fell back to the reference object, False if your inputs were present.

    The reference archive for a stage is cumulative: it holds every earlier stage folder that
    later stages read, plus ground_truth/, so a fallback gives one consistent object (see
    _extract_reference for what happens to folders of yours). In stages 5 and 6 it then makes
    sure that a reference object also gets the reference capture/ (ensure_reference_capture).
    """
    prev = prev_folder(stage)
    folder = PROJECT / prev
    fell_back = not (folder.is_dir() and any(folder.iterdir()))
    if fell_back:
        archive = "stage0.tar.gz" if stage == 1 else f"{prev}.tar.gz"
        print(f"{prev}/ is missing or empty: running on the reference object ({archive}).")
        _extract_reference(archive)
    ensure_reference_capture(stage)
    return fell_back


def ensure_generated(obj):
    """Fetch the course's TRELLIS.2 archive for a reference object and return its folder.

    generated-<obj>.tar.gz holds that object's stage folders, ground_truth/ and trellis2/. They
    go to generated/<obj>/, which is replaced whole on each download, so your own stage folders
    are never touched. A failed download leaves the previous generated/<obj>/ as it was. When
    generated/<obj>/trellis2/trellis2.json is already there with GENERATED_MODEL_REVISION, the
    folder is kept and nothing is downloaded.
    """
    if obj not in GENERATED_OBJECTS:
        raise ValueError(f"no generated archive for {obj!r}; the course ships {', '.join(GENERATED_OBJECTS)}")
    out = PROJECT / "generated" / obj
    try:
        revision = json.loads((out / "trellis2" / "trellis2.json").read_text()).get("model_revision")
    except (OSError, ValueError, AttributeError):
        revision = None
    if revision == GENERATED_MODEL_REVISION:
        print(f"generated/{obj}/ is already here (TRELLIS.2 revision {revision[:7]}), so it is not downloaded again.")
        return out
    with atomic_dir(out) as tmp:
        _extract_reference(f"generated-{obj}.tar.gz", dest=tmp)
    return out


import rerun as rr

RRD_PATH = PROJECT / "rrd" / f"stage{STAGE}.rrd"
RRD_PATH.parent.mkdir(parents=True, exist_ok=True)
rr.init("real-to-sim-twin", recording_id=RECORDING_ID)
rr.save(RRD_PATH)
rr.log("world", rr.ViewCoordinates.RIGHT_HAND_Z_UP, static=True)


def show_rerun():
    """Show everything logged so far in an inline viewer. Call it last in the stage.

    The viewer replaces the file sink, which leaves stage<N>.rrd complete; the viewer only gets
    data logged after it attaches, so the file is replayed into it. Later log calls reach the
    viewer only, not the file.
    """
    rr.notebook_show()
    rr.log_file_from_path(RRD_PATH)

In this section you give the twin the properties a simulator needs to move it. A rigid body in a physics engine is described by its mass, the position of its center of mass, its inertia tensor about that point, the shapes it collides with, and the friction between those shapes and the ground. The mesh from stage 3 supplies the geometry. The rest comes from your measurements in `physical.json`: the mass from a kitchen scale, five tilt angles at which the object starts to slide, three dimensions taken with a ruler, and three readings of where the object balances along its long axis.

The collision shapes come from CoACD (Wei et al., "Approximate Convex Decomposition for 3D Meshes with Collision-Aware Concavity and Tree Search", 2022, MIT license), which splits the mesh into a small number of convex pieces. Volumes and inertia tensors are integrated over the closed mesh with trimesh (MIT license).

Runtime budget on a Colab T4: about 5 minutes, the first run in a new session included, about 4 of them in the convex decomposition. The stage runs on the CPU alone.


## Your measurements: `capture/physical.json`

Before you run this stage, write your measurements to `capture/physical.json` in your project folder. The next cell reads the file and stops with an error when a key is missing. The values below are an example for a small wooden-handled tool, not a reference.

```json
{
  "mass_kg": 0.412,
  "tilt_deg": [24.0, 25.5, 24.5, 25.0, 24.0],
  "dims_mm": {"length": 182.0, "width": 74.5, "height": 31.0},
  "balance": {"readings_mm": [121.0, 123.5, 122.0], "end": "handle"},
  "materials": {"metal": {"name": "steel", "density_kg_m3": 7850},
                "other": {"name": "beech wood", "density_kg_m3": 700}}
}
```

- `mass_kg`: the object on a kitchen scale, in kilograms.
- `tilt_deg`: lay a hardback book or a stiff board flat with the object on it, resting the way it rests on a table, and put your phone's level app on the board next to it. Raise one end slowly and read the angle at which the object starts to slide. Do this five times, putting the object back in the same place each time. If it tips over or rolls before it slides, turn it until it slides; Question 2 below explains why.
- `dims_mm`: three sizes in millimeters, taken with a ruler: the longest side of the object and the two sides across it. The names are yours. The check compares the three values, sorted, with the mesh's box.
- `balance`: lay the object across a pencil, perpendicular to its long axis, and move it until it balances. Measure the distance from one end of the object to the pencil, three times, and name that end in `end` with a word such as `"bottom"`, `"top"`, `"handle"` or `"head"`. Later in this stage you set `END_SIGN` to say which end of the mesh's long axis that is.
- `materials`: only for an object made of a metal part and a non-metal part, such as a hammer or a knife with a wooden handle. Give each part's name and its handbook density in kg/m³. Leave the key out for an object of one material.

If one of the values is an estimate rather than a measurement, add `"assumed": true` and a `"note"` that says which, and the stage prints it with the results.

In [ ]:
import json
import time

import numpy as np
import trimesh

T0 = time.time()
FALLBACK = ensure_inputs(STAGE)
STAGE3, STAGE4 = PROJECT / "stage3", PROJECT / "stage4"
REFERENCE = on_reference(STAGE)  # the reference stage4/ carries a .reference marker
physical = json.loads((PROJECT / "capture" / "physical.json").read_text())
mesh = trimesh.load(STAGE3 / "mesh.ply", process=False)
SCAN_OBJECT = json.loads((PROJECT / "ground_truth" / "scan.json").read_text())["object"] if REFERENCE else None
print("object:", f"reference ({SCAN_OBJECT})" if REFERENCE else OBJECT)
print(f"stage 3 mesh: {len(mesh.faces):,} triangles, watertight {mesh.is_watertight}")
print(f"physical.json: mass {physical['mass_kg']} kg, tilt readings {physical['tilt_deg']} degrees, "
      f"dimensions {physical['dims_mm']} mm")
if "balance" in physical:
    print(f"balance readings {physical['balance']['readings_mm']} mm from the {physical['balance']['end']} end")
print("materials:", ", ".join(f"{k} = {v['name']}" for k, v in physical["materials"].items())
      if "materials" in physical else "none given (one density)")
if physical.get("assumed"):
    print("These values are assumed, not measured:", physical.get("note", ""))

## Convex pieces for collision

A simulator tests every pair of shapes for contact at every step, often a thousand times per simulated second. For two convex shapes the test is fast and well defined. Two disjoint convex sets are always separated by a plane, and an algorithm such as GJK finds their closest points by asking each shape only for its farthest vertex in a given direction. The cost grows with the vertex counts, not with the triangle counts, and the answer is one contact normal and one depth. A non-convex mesh has neither property. Two of them can touch at many places at once, and the contact is found by testing triangles against triangles, which is slow for a mesh of a quarter of a million triangles and gives contact normals that jump from one triangle to the next.

Simulators therefore cover the object with a few convex pieces. The convex hull of the whole mesh is the simplest cover. It fills every concave region, so a mug becomes a solid cylinder and a hammer becomes a wedge that spans its head and handle. CoACD cuts the mesh with planes, searching over cuts with a tree search, until every piece is close to its own convex hull. It measures closeness with a concavity that combines the distance from the piece's surface to its hull and the depth a colliding object could sink into the gap. The cell below stops cutting when the concavity of every piece is below 0.05 of the mesh's bounding box, and then merges pieces until at most 32 remain, because each extra piece costs contact tests in the simulator. The threshold is relative to the bounding box, so on a long object it allows deep concavity. On a hammer about 34 cm long, 0.05 lets the pieces bridge the corner where the handle meets the head, and their union comes out 13% larger than the mesh. When the union is more than 10% larger than the mesh, the cell halves the threshold and runs CoACD again with the same cap, down to 0.01, and prints each attempt.

The cell prints three volumes next to the mesh's own. The union of the pieces is what the simulator will treat as solid. The sum of the piece volumes counts their overlaps twice. The single convex hull shows what the decomposition improves on.


In [ ]:
import numpy as np
import trimesh


def decompose(mesh, threshold: float = 0.05, max_pieces: int = 32, seed: int = 0) -> list:
    """Approximate convex decomposition with CoACD. Returns the pieces as convex trimesh meshes.

    threshold is CoACD's concavity limit, relative to the mesh's bounding box; max_pieces caps
    the count by merging the pieces whose merge adds the least concavity. Each piece is returned as
    the convex hull of CoACD's output, so its vertex order is the hull's, not CoACD's.
    """
    import coacd

    coacd.set_log_level("error")
    parts = coacd.run_coacd(coacd.Mesh(np.asarray(mesh.vertices), np.asarray(mesh.faces)), threshold=threshold,
                            max_convex_hull=max_pieces, seed=seed)
    return [trimesh.Trimesh(v, f, process=False).convex_hull for v, f in parts]


def decompose_within(mesh, mesh_volume: float, threshold: float = 0.05, max_pieces: int = 32,
                     volume_tol: float = 0.10, min_threshold: float = 0.01) -> tuple[list, float, list]:
    """decompose, halving the threshold until the union of the pieces is within volume_tol of
    mesh_volume or the next threshold would fall below min_threshold (CoACD's lowest).

    The threshold is relative to the bounding box, so on a long object it allows deep concavity.
    Returns (pieces, union volume, attempts) for the attempt closest to mesh_volume; attempts lists
    every run as {"threshold", "pieces", "union", "seconds"}.
    """
    import time

    attempts, best = [], None
    while True:
        t = time.time()
        pieces = decompose(mesh, threshold=threshold, max_pieces=max_pieces)
        union = union_volume(pieces)
        attempts.append({"threshold": threshold, "pieces": len(pieces), "union": union, "seconds": time.time() - t})
        if best is None or abs(union / mesh_volume - 1) < abs(best[1] / mesh_volume - 1):
            best = (pieces, union)
        if abs(union / mesh_volume - 1) <= volume_tol or threshold / 2 < min_threshold:
            return best[0], best[1], attempts
        threshold /= 2


def inside_pieces(points, pieces, tol: float = 1e-9) -> np.ndarray:
    """For each point, True if it lies inside at least one convex piece."""
    points = np.asarray(points, dtype=float)
    inside = np.zeros(len(points), bool)
    for p in pieces:
        lo, hi = p.bounds
        sel = np.nonzero(np.all((points >= lo) & (points <= hi), axis=1) & ~inside)[0]
        if len(sel):
            dist = np.einsum("nd,fd->nf", points[sel], p.face_normals) - np.einsum(
                "fd,fd->f", p.triangles[:, 0, :], p.face_normals)
            inside[sel[np.all(dist <= tol, axis=1)]] = True
    return inside


def union_volume(pieces, n: int = 400_000, seed: int = 0) -> float:
    """Volume of the union of the pieces (m^3), by uniform random points in their joint box."""
    lo = np.min([p.bounds[0] for p in pieces], axis=0)
    hi = np.max([p.bounds[1] for p in pieces], axis=0)
    points = np.random.default_rng(seed).uniform(lo, hi, size=(n, 3))
    return float(inside_pieces(points, pieces).mean() * np.prod(hi - lo))

In [ ]:
t = time.time()
mesh_volume = mesh.volume  # divergence theorem over the closed mesh
pieces, pieces_union, attempts = decompose_within(mesh, mesh_volume, threshold=0.05, max_pieces=32)
decompose_s = time.time() - t
for a in attempts:
    print(f"CoACD at threshold {a['threshold']:.4f}: {a['pieces']} pieces in {a['seconds']:.0f} s, "
          f"union {a['union'] / mesh_volume - 1:+.1%} off the mesh volume")
print(f"CoACD: {len(pieces)} convex pieces in {decompose_s:.0f} s, "
      f"{min(len(p.faces) for p in pieces)} to {max(len(p.faces) for p in pieces)} triangles each")
print(f"volume: mesh {mesh_volume * 1e6:.1f} mL, union of the pieces {pieces_union * 1e6:.1f} mL "
      f"({pieces_union / mesh_volume - 1:+.1%}), sum of the pieces {sum(p.volume for p in pieces) * 1e6:.1f} mL, "
      f"one convex hull of the whole mesh {mesh.convex_hull.volume * 1e6:.1f} mL")

## Mass, center of mass and inertia

With a density $\rho$ that is the same everywhere, the mass, the center of mass and the inertia tensor about it are integrals over the volume $V$ enclosed by the mesh:

$$
m = \rho V, \qquad c = \frac{1}{V} \int_V x \, dV, \qquad I = \rho \int_V \left( \lVert r \rVert^2 E - r r^\top \right) dV, \quad r = x - c,
$$

where $E$ is the 3 by 3 identity. The divergence theorem turns each volume integral into a sum over the triangles of the closed surface: every triangle and the origin span a tetrahedron, whose signed volume and moments have closed forms, and the tetrahedra of the far side of the mesh cancel the parts outside it. The integrals are exact for the mesh. They need a closed surface, which is why stage 3 closed the unseen bottom of the object with a flat cap on the support plane. Volume, mass distribution and the balance prediction below all inherit that flat base, so an object with a hollow or domed base gets a center of mass slightly lower than its own.

You do not know $\rho$, but you know $m$ from the scale, so $\rho = m / V$. The inertia then follows from the mesh at unit density, scaled by $\rho$. The diagonal entries of $I$ are the moments about the three coordinate axes, and the off-diagonal entries measure how far those axes are from the object's own axes of symmetry. The eigenvalues of $I$ are the principal moments. For a solid box with sides $a$, $b$, $c$ and mass $m$ they are

$$
I_a = \frac{m}{12} \left( b^2 + c^2 \right), \quad I_b = \frac{m}{12} \left( a^2 + c^2 \right), \quad I_c = \frac{m}{12} \left( a^2 + b^2 \right),
$$

and the cell compares them with your object's, for a box with your object's mass and oriented bounding box. A ratio below 1 means that the object keeps less of its mass far from its center than the box does, through rounded edges, a tapered end or a concave side.


In [ ]:
import numpy as np


def inertia_scaled(mesh, mass_kg: float) -> np.ndarray:
    """Inertia tensor (3 x 3, kg m^2) about the center of mass, for uniform density and the given mass.

    mesh is a closed trimesh mesh in meters. trimesh's mesh.moment_inertia is the inertia
    tensor about mesh.center_mass at the density mesh.density (1 kg/m^3 unless set), and
    mesh.volume is the enclosed volume. Scale the tensor so that the density is
    mass_kg / mesh.volume.
    """
    raise NotImplementedError

In [ ]:
mass = float(physical["mass_kg"])
density = mass / mesh_volume
com_uniform = np.asarray(mesh.center_mass)
I_uniform = inertia_scaled(mesh, mass)
to_obb, extents = trimesh.bounds.oriented_bounds(mesh)  # to_obb maps the mesh into its box's frame
principal = np.sort(np.linalg.eigvalsh(I_uniform))
a, b, c = extents
box_moments = np.sort(mass / 12 * np.array([b**2 + c**2, a**2 + c**2, a**2 + b**2]))
print(f"uniform density {density:,.0f} kg/m^3 from {mass} kg over {mesh_volume * 1e6:.1f} mL")
print(f"center of mass {np.round(com_uniform * 1e3, 2)} mm")
print("inertia about the center of mass (kg m^2):\n", np.array2string(I_uniform, precision=3))
print(f"principal moments {np.round(principal * 1e6, 1)} x 1e-6 kg m^2; a solid box of the same mass and its "
      f"bounding box ({np.round(extents * 1e3, 1)} mm): {np.round(box_moments * 1e6, 1)} x 1e-6 kg m^2 "
      f"(ratio {np.round(principal / box_moments, 3)})")

## A second density

Uniform density is a fair model of a carton of milk or a block of wood. It is wrong for an object made of two materials, such as a hammer with a steel head on a wooden handle. Steel is about 13 times as dense as wood, so the head holds most of the mass, and a uniform model puts the center of mass far down the handle.

The two-density model decides which parts of the object are metal from the metallic map of stage 4, one convex piece at a time. Each piece takes the mean metallic value of the mesh surface that falls inside it, and a piece whose mean is above 0.5 is metal. The pieces only label the volume. The mass is still integrated over the closed mesh, exactly the volume the uniform model uses, so the two models differ in density and in nothing else. The interior of the mesh is filled with a regular grid of small cells, about a million of them, cut off where they cross the surface. Each cell takes the material of the piece that contains its center (where pieces overlap, or where a cell falls in no piece, the piece with the nearest center). With densities $\rho_j$ from `physical.json` (handbook values, for example steel 7850 kg/m³ and wood 600 kg/m³), cell $j$ with volume $v_j$ and center $x_j$ has mass $m_j = \rho_j v_j$, and

$$
c = \frac{\sum_j m_j x_j}{\sum_j m_j}, \qquad
I = \sum_j m_j \left( \lVert r_j \rVert^2 E - r_j r_j^\top \right), \quad r_j = x_j - c .
$$

Each cell counts as a point mass at its center. By the parallel axis theorem, a cell of side $h$ also carries its own inertia $m_j h^2 / 6$ about its center, but for cells under a millimeter on an object several centimeters across that term is far below one part in a thousand of $I$. The handbook densities rarely give exactly the mass on your scale, so both densities are multiplied by one factor $k = m / \sum_j m_j$. Scaling every mass by $k$ leaves $c$ unchanged and multiplies $I$ by $k$. Only the ratio of the two densities shapes the result, and the scale fixes the total.

The model runs only when `physical.json` names two materials, because the metallic map can be wrong. On the reference carton, which is printed paperboard, the map puts 5 of the 32 pieces above 0.5, about 10% of the carton's volume, and a two-density model would fill that tenth with steel. The signal is weak on real metal too. On a hammer with a bare steel head and a wooden handle, the head's surface reads about 0.50 on average and the handle's about 0.42, so the head sits right at the threshold. A small shift in the map can then move a piece from one material to the other and flip which end of the model is heavy. The cell prints the metallic value of each piece and the share of the volume it would call metal, so you can judge the map on your own object before you trust it.


In [ ]:
import cv2
import numpy as np
from pathlib import Path

DEFAULT_DENSITY = {"steel": 7850.0, "iron": 7870.0, "aluminium": 2700.0, "aluminum": 2700.0, "brass": 8500.0,
                   "copper": 8960.0, "wood": 600.0, "plastic": 1000.0, "rubber": 1100.0, "glass": 2500.0}


def load_mesh_uv(path: Path) -> dict:
    """Read stage4/mesh_uv.obj: "vertices" (V x 3, meters), "faces" (F x 3) and "uvs" (V x 2,
    origin at the top-left of the texture, v down; the OBJ stores v up)."""
    v, vt, f = [], [], []
    for line in Path(path).read_text().splitlines():
        if line.startswith("v "):
            v.append(line.split()[1:4])
        elif line.startswith("vt "):
            vt.append(line.split()[1:3])
        elif line.startswith("f "):
            f.append([int(w.split("/")[0]) - 1 for w in line.split()[1:4]])
    uvs = np.asarray(vt, dtype=float)
    if len(uvs) != len(v):  # stage 4 writes f a/a/a: vertex i has texture coordinate i
        raise ValueError(f"{path} has {len(v)} vertices and {len(uvs)} texture coordinates; stage 5 reads the "
                         "stage 4 format, one texture coordinate per vertex")
    return {"vertices": np.asarray(v, dtype=float), "faces": np.asarray(f, dtype=np.int64),
            "uvs": np.c_[uvs[:, 0], 1 - uvs[:, 1]]}


def surface_samples(mesh_uv: dict, texture: np.ndarray, n: int = 200_000, seed: int = 0):
    """n points spread uniformly by area over the mesh, and the texture value at each (nearest texel)."""
    rng = np.random.default_rng(seed)
    tri = mesh_uv["vertices"][mesh_uv["faces"]]
    area = 0.5 * np.linalg.norm(np.cross(tri[:, 1] - tri[:, 0], tri[:, 2] - tri[:, 0]), axis=1)
    face = rng.choice(len(area), size=n, p=area / area.sum())
    r1, r2 = rng.random(n), rng.random(n)
    s = np.sqrt(r1)
    w = np.c_[1 - s, s * (1 - r2), s * r2]  # uniform barycentric weights
    points = np.einsum("nk,nkd->nd", w, tri[face])
    uv = np.einsum("nk,nkd->nd", w, mesh_uv["uvs"][mesh_uv["faces"][face]])
    h, wd = texture.shape[:2]
    rows = np.clip((uv[:, 1] * h).astype(int), 0, h - 1)
    cols = np.clip((uv[:, 0] * wd).astype(int), 0, wd - 1)
    return points, texture[rows, cols]


def assign_to_pieces(points, pieces) -> np.ndarray:
    """Index of the piece each point belongs to: the piece it lies deepest inside, or, for a point
    outside all of them, the one it lies closest outside (largest face-plane distance smallest)."""
    best = np.full(len(points), np.inf)
    owner = np.zeros(len(points), dtype=int)
    for i, p in enumerate(pieces):
        offsets = np.einsum("fd,fd->f", p.triangles[:, 0, :], p.face_normals)
        for start in range(0, len(points), 50_000):
            chunk = points[start:start + 50_000]
            out = (chunk @ p.face_normals.T - offsets).max(axis=1)  # < 0 inside, > 0 outside
            better = out < best[start:start + 50_000]
            best[start:start + 50_000][better] = out[better]
            owner[start:start + 50_000][better] = i
    return owner


def piece_metallic(mesh_uv: dict, metallic: np.ndarray, pieces, n: int = 200_000, seed: int = 0) -> np.ndarray:
    """Mean metallic value (0 to 1) of the mesh surface that falls inside each convex piece.

    A piece that holds no sample (a sliver) takes the value of the nearest sample to its center.
    """
    points, values = surface_samples(mesh_uv, metallic, n, seed)
    owner = assign_to_pieces(points, pieces)
    out = np.zeros(len(pieces))
    for i, p in enumerate(pieces):
        mine = owner == i
        out[i] = values[mine].mean() if mine.any() else values[np.argmin(
            np.linalg.norm(points - p.center_mass, axis=1))]
    return out


def interior_points(mesh, n: int = 1_000_000):
    """Sample points filling the closed mesh, and the volume (m^3) each one stands for.

    A regular grid of vertical columns, pitch h, set so that about n cells fall inside. Each
    column's line crosses the surface at entry and exit heights (sorted crossings, taken in
    pairs), and the column is cut into cells of height h between them. A cell cut by an entry or
    an exit keeps only its inside part: its point sits at the middle of that part, and its volume
    is h^2 times its length. The grid is shifted by a small irrational fraction of h so that no
    line runs exactly along a triangle edge.
    """
    h = (mesh.volume / n) ** (1 / 3)
    lo, hi = mesh.bounds
    origin = lo + h * np.array([0.5 + 1e-3 * np.sqrt(2), 0.5 + 1e-3 * np.sqrt(3), 0.5])
    counts = np.floor((hi - origin) / h).astype(int) + 1
    tri = np.asarray(mesh.triangles)
    i0 = np.ceil((tri[:, :, 0].min(1) - origin[0]) / h).astype(int)
    i1 = np.floor((tri[:, :, 0].max(1) - origin[0]) / h).astype(int)
    j0 = np.ceil((tri[:, :, 1].min(1) - origin[1]) / h).astype(int)
    j1 = np.floor((tri[:, :, 1].max(1) - origin[1]) / h).astype(int)
    ni, nj = np.clip(i1 - i0 + 1, 0, None), np.clip(j1 - j0 + 1, 0, None)
    cnt = ni * nj
    f = np.repeat(np.arange(len(tri)), cnt)  # one entry per (triangle, grid column under its xy box)
    k = np.arange(len(f)) - np.repeat(np.cumsum(cnt) - cnt, cnt)
    ii, jj = i0[f] + k // nj[f], j0[f] + k % nj[f]
    px, py = origin[0] + ii * h, origin[1] + jj * h
    a, b, c = tri[f, 0], tri[f, 1], tri[f, 2]
    det = (b[:, 0] - a[:, 0]) * (c[:, 1] - a[:, 1]) - (c[:, 0] - a[:, 0]) * (b[:, 1] - a[:, 1])
    ok = np.abs(det) > 1e-18  # a vertical triangle is never crossed by a vertical line
    det = np.where(ok, det, 1.0)
    w1 = ((px - a[:, 0]) * (c[:, 1] - a[:, 1]) - (c[:, 0] - a[:, 0]) * (py - a[:, 1])) / det
    w2 = ((b[:, 0] - a[:, 0]) * (py - a[:, 1]) - (px - a[:, 0]) * (b[:, 1] - a[:, 1])) / det
    w0 = 1 - w1 - w2
    hit = ok & (w0 >= 0) & (w1 >= 0) & (w2 >= 0)
    col = (ii * counts[1] + jj)[hit]
    z = (w0 * a[:, 2] + w1 * b[:, 2] + w2 * c[:, 2])[hit]
    order = np.lexsort((z, col))
    col, z = col[order], z[order]
    _, first, per_col = np.unique(col, return_index=True, return_counts=True)
    even = np.repeat(per_col % 2 == 0, per_col)  # a column with an odd count grazed an edge: skip it
    col, z = col[even], z[even]
    z_in, z_out, col = z[0::2], z[1::2], col[0::2]
    k0 = np.floor((z_in - origin[2]) / h + 0.5).astype(int)  # the cells holding the entry and the exit
    k1 = np.floor((z_out - origin[2]) / h + 0.5).astype(int)
    m = k1 - k0 + 1
    seg = np.repeat(np.arange(len(m)), m)
    kk = k0[seg] + np.arange(m.sum()) - np.repeat(np.cumsum(m) - m, m)
    bottom = np.maximum(origin[2] + (kk - 0.5) * h, z_in[seg])
    top = np.minimum(origin[2] + (kk + 0.5) * h, z_out[seg])
    keep = top > bottom
    cc, seg, bottom, top = col[seg][keep], seg[keep], bottom[keep], top[keep]
    points = np.c_[origin[0] + h * (cc // counts[1]), origin[1] + h * (cc % counts[1]), (bottom + top) / 2]
    return points, h * h * (top - bottom)


def owning_piece(points, pieces) -> np.ndarray:
    """Index of the convex piece each point belongs to: the piece that contains it, or, when
    several or none contain it, the piece whose center of mass is nearest (among the containing
    ones if there are several)."""
    points = np.asarray(points, dtype=float)
    inside = np.zeros((len(pieces), len(points)), bool)
    for i, p in enumerate(pieces):
        lo, hi = p.bounds
        sel = np.nonzero(np.all((points >= lo) & (points <= hi), axis=1))[0]
        if len(sel):
            offsets = np.einsum("fd,fd->f", p.triangles[:, 0, :], p.face_normals)
            inside[i, sel] = np.all(points[sel] @ p.face_normals.T - offsets <= 1e-9, axis=1)
    owner = inside.argmax(axis=0)
    unclear = np.nonzero(inside.sum(axis=0) != 1)[0]
    centers = np.array([p.center_mass for p in pieces])
    for start in range(0, len(unclear), 50_000):
        idx = unclear[start:start + 50_000]
        d = np.linalg.norm(points[idx, None, :] - centers[None], axis=2)
        several = inside[:, idx].T
        d = np.where(several.any(axis=1, keepdims=True) & ~several, np.inf, d)
        owner[idx] = d.argmin(axis=1)
    return owner


def densities_from_materials(materials: dict) -> tuple[float, float]:
    """(rho_metal, rho_other) in kg/m^3 from physical.json "materials". A missing density is looked
    up by name (steel 7850, wood 600 and a few others)."""
    rho = []
    for key in ("metal", "other"):
        m = materials[key]
        if m.get("density_kg_m3"):
            rho.append(float(m["density_kg_m3"]))
            continue
        hits = [d for name, d in DEFAULT_DENSITY.items() if name in m["name"].lower()]
        if not hits:
            raise ValueError(f'materials["{key}"] = {m["name"]!r} has no density_kg_m3 and no default; add one')
        rho.append(hits[0])
    return rho[0], rho[1]

In [ ]:
import numpy as np


def mass_properties_two_density(mesh, pieces, piece_metallic: np.ndarray, rho_metal: float, rho_other: float,
                                mass_kg: float) -> tuple[np.ndarray, np.ndarray]:
    """Center of mass (3,) and inertia about it (3 x 3) of the closed mesh filled with two densities.

    1. points, volumes = interior_points(mesh) gives points filling the mesh and the volume each
       stands for. owning_piece(points, pieces) gives the convex piece each point belongs to; the
       point is metal if that piece's piece_metallic value is above 0.5.
    2. Give each point the mass m_j = rho_j * volumes[j], with rho_j = rho_metal or rho_other.
    3. The center of mass is sum(m_j x_j) / sum(m_j).
    4. The inertia about it is the sum over points of m_j (|r_j|^2 E - r_j r_j^T), r_j = x_j - com
       (each point is a point mass; a cell's own inertia is negligible at this cell size).
    5. Scaling both densities by k = mass_kg / sum(m_j) leaves the center unchanged and
       multiplies the inertia by k.
    """
    raise NotImplementedError

In [ ]:
mesh_uv = load_mesh_uv(STAGE4 / "mesh_uv.obj")
metallic_map = cv2.imread(str(STAGE4 / "metallic.png"), cv2.IMREAD_GRAYSCALE) / 255.0
metal_of_piece = piece_metallic(mesh_uv, metallic_map, pieces)
metal = metal_of_piece > 0.5
grid, volumes = interior_points(mesh)
metal_cells = metal[owning_piece(grid, pieces)]
print(f"stage 4 metallic per piece: mean {metal_of_piece.mean():.3f}, range {metal_of_piece.min():.3f} to "
      f"{metal_of_piece.max():.3f}; {metal.sum()} of {len(pieces)} pieces above 0.5, which hold "
      f"{volumes[metal_cells].sum() / volumes.sum():.0%} of the object's volume")
print("metallic of each piece, in piece order:", " ".join(f"{m:.2f}" for m in metal_of_piece))
cell_mm = volumes.max() ** (1 / 3) * 1e3  # a whole cell is h^3
print(f"interior points: {len(grid):,} in cells of {cell_mm:.2f} mm, {volumes.sum() * 1e6:.1f} mL against the "
      f"mesh's {mesh_volume * 1e6:.1f} mL")
if "materials" in physical:
    rho_metal, rho_other = densities_from_materials(physical["materials"])
    com_two, I_two = mass_properties_two_density(mesh, pieces, metal_of_piece, rho_metal, rho_other, mass)
    k = mass / (np.where(metal_cells, rho_metal, rho_other) @ volumes)
    print(f"two densities: {physical['materials']['metal']['name']} {rho_metal:,.0f} and "
          f"{physical['materials']['other']['name']} {rho_other:,.0f} kg/m^3, scaled by {k:.3f} to "
          f"{k * rho_metal:,.0f} and {k * rho_other:,.0f} kg/m^3 so the total is {mass} kg")
    print(f"center of mass {np.round(com_two * 1e3, 2)} mm, moved {np.linalg.norm(com_two - com_uniform) * 1e3:.1f} mm "
          f"from the uniform one")
    print("inertia about it (kg m^2):\n", np.array2string(I_two, precision=3))
else:
    com_two = I_two = None
    print("physical.json names no materials, so the two-density model is skipped and stage 5 uses one density.")

## Balance point

A center of mass is hard to measure directly, but its projection on one axis is easy. Lay the object across a pencil or the edge of a table, perpendicular to its long axis, and move it until it balances. The support then sits under the center of mass, and its distance from a named end of the object is the coordinate of the center of mass along the long axis, measured from that end.

The long axis of the mesh is the longest axis of its oriented bounding box, the smallest box around the mesh with any orientation. Its direction $a$ is a unit vector through the box center $o$, and the mesh spans the coordinates $s_- \le s \le s_+$ along it, with the outermost 0.1% of the surface left out at each end as in the dimension check below. A center of mass $c$ projects to $s_c = (c - o) \cdot a$, so its distance from the $+$ end is $s_+ - s_c$ and from the $-$ end is $s_c - s_-$. The figure marks the two ends. Set `END_SIGN` in the cell below to $+1$ if the end you named in `physical.json` is the $+$ end and to $-1$ if it is the $-$ end, then run it. On the reference object the cell sets `END_SIGN` itself, from the end that `physical.json` names and the shape of the mesh. The cell prints each model's prediction next to the mean and the spread of your three readings.


In [ ]:
import numpy as np


def long_axis(mesh, to_obb: np.ndarray, extents, trim: float = 0.001, n: int = 400_000, seed: int = 0) -> dict:
    """The mesh's long axis: the major axis of its oriented bounding box, through the box center.

    The axis points so that its largest component is positive (an upright object's axis points
    up). Returns "center", "axis" (unit), and "lo", "hi": the coordinates along the axis of the
    mesh's two ends, measured from the center (the - end and the + end). Like the dimensions
    check, the ends leave out the outermost `trim` share of the surface area, so that a bump of
    the reconstruction does not move an end.
    """
    import trimesh

    from_obb = np.linalg.inv(to_obb)
    k = int(np.argmax(extents))
    axis = from_obb[:3, k] / np.linalg.norm(from_obb[:3, k])
    axis = axis if axis[np.argmax(np.abs(axis))] > 0 else -axis
    center = from_obb[:3, 3]
    samples, _ = trimesh.sample.sample_surface(mesh, n, seed=seed)
    s = (samples - center) @ axis
    return {"center": center, "axis": axis, "lo": float(np.percentile(s, 100 * trim)),
            "hi": float(np.percentile(s, 100 * (1 - trim)))}


END_NAMES = {"bottom": "low", "base": "low", "top": "high", "handle": "narrow", "narrow": "narrow",
             "head": "wide", "wide": "wide"}


def end_sign_from_name(name: str, mesh, axis_info: dict, part: float = 0.12, n: int = 200_000, seed: int = 0) -> int:
    """The sign (+1 or -1) of the end of the long axis that a name in physical.json describes.

    "bottom" or "base" is the lower end and "top" the higher one (world z); "handle" or "narrow" is
    the end whose last `part` of the length is thinner (95th percentile distance from the axis),
    "head" or "wide" the thicker one. Other names raise ValueError: set END_SIGN by hand.
    """
    import trimesh

    kind = END_NAMES.get(name.strip().lower())
    if kind is None:
        raise ValueError(f"no rule maps the end name {name!r} to a side of the long axis: set END_SIGN by hand")
    c, a, lo, hi = axis_info["center"], axis_info["axis"], axis_info["lo"], axis_info["hi"]
    if kind in ("low", "high"):
        plus_higher = (c + hi * a)[2] > (c + lo * a)[2]
        return 1 if plus_higher == (kind == "high") else -1
    samples, _ = trimesh.sample.sample_surface(mesh, n, seed=seed)
    s = (samples - c) @ a
    r = np.linalg.norm(samples - c - np.outer(s, a), axis=1)
    length = hi - lo
    plus = np.percentile(r[s > hi - part * length], 95)
    minus = np.percentile(r[s < lo + part * length], 95)
    plus_narrower = plus < minus
    return 1 if plus_narrower == (kind == "narrow") else -1


def balance_from_end(com, axis_info: dict, end_sign: int) -> float:
    """Distance (m) from the named end to the projection of com on the long axis.

    end_sign is +1 if the named end is the + end of the axis, -1 if it is the - end.
    """
    s = float((np.asarray(com) - axis_info["center"]) @ axis_info["axis"])
    return axis_info["hi"] - s if end_sign > 0 else s - axis_info["lo"]

In [ ]:
axis_info = long_axis(mesh, to_obb, extents)
print(f"long axis {np.round(axis_info['axis'], 3)} through {np.round(axis_info['center'] * 1e3, 1)} mm; "
      f"the mesh spans {(axis_info['hi'] - axis_info['lo']) * 1e3:.1f} mm along it")

In [ ]:
import io

import matplotlib.pyplot as plt
from IPython.display import Image as IPImage, display
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

colors = plt.cm.tab20(np.arange(len(pieces)) % 20)
length = axis_info["hi"] - axis_info["lo"]
ends = {"+": axis_info["center"] + axis_info["hi"] * axis_info["axis"],
        "-": axis_info["center"] + axis_info["lo"] * axis_info["axis"]}
beyond = {k: axis_info["center"] + (axis_info["hi"] + 0.2 * length if k == "+" else axis_info["lo"] - 0.2 * length)
          * axis_info["axis"] for k in ends}
fig = plt.figure(figsize=(12, 5.6))
for n, (title, face_colors) in enumerate([("convex pieces", colors),
                                         ("metallic per piece (dark 0, bright 1)", plt.cm.magma(metal_of_piece))]):
    ax = fig.add_subplot(1, 2, n + 1, projection="3d", computed_zorder=False)
    for p, col in zip(pieces, face_colors):
        ax.add_collection3d(Poly3DCollection(p.triangles * 1e3, facecolor=col, edgecolor="none", alpha=0.3, zorder=1))
    line = np.array([beyond["-"], beyond["+"]]) * 1e3
    ax.plot(*line.T, color="black", lw=2, zorder=10)
    for name, pos in beyond.items():
        ax.text(*(pos * 1e3), f" {name} end", fontsize=12, weight="bold", zorder=11)
    ax.scatter(*(com_uniform * 1e3), color="black", s=50, zorder=12, label="center of mass, uniform")
    if com_two is not None:
        ax.scatter(*(com_two * 1e3), color="red", s=50, zorder=12, label="center of mass, two densities")
    lo, hi = np.r_[mesh.bounds, line / 1e3].min(0) * 1e3, np.r_[mesh.bounds, line / 1e3].max(0) * 1e3
    mid, r = (lo + hi) / 2, (hi - lo).max() / 2
    ax.set(xlim=(mid[0] - r, mid[0] + r), ylim=(mid[1] - r, mid[1] + r), zlim=(mid[2] - r, mid[2] + r),
           xlabel="x (mm)", ylabel="y (mm)", zlabel="z (mm)", title=title)
    ax.view_init(elev=20, azim=-60)
    ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
buf = io.BytesIO()
fig.savefig(buf, format="jpeg", dpi=70, pil_kwargs={"quality": 85})
plt.close(fig)
display(IPImage(data=buf.getvalue()))

In [ ]:
# Set END_SIGN from the figure above (or the viewer below): +1 if the end named in physical.json
# ("end") is the + end of the long axis, -1 if it is the - end.
END_SIGN = None
balance = physical.get("balance")
if END_SIGN is None and REFERENCE and balance:  # the reference object: from the named end and the mesh
    END_SIGN = end_sign_from_name(balance["end"], mesh, axis_info)
    print(f"reference object: its {balance['end']} end is the {'+' if END_SIGN > 0 else '-'} end, END_SIGN = {END_SIGN}")
if balance:
    assert END_SIGN in (1, -1), "Set END_SIGN to +1 or -1 from the figure above."
else:
    END_SIGN = END_SIGN or -1  # no balance readings: the predictions are measured from the - end
predicted = {"uniform": balance_from_end(com_uniform, axis_info, END_SIGN)}
if com_two is not None:
    predicted["two_density"] = balance_from_end(com_two, axis_info, END_SIGN)
if balance:
    readings = np.asarray(balance["readings_mm"], dtype=float)
    measured_mm, spread_mm = readings.mean(), readings.std(ddof=1)
    print(f"balance point from the {balance['end']} end: measured {measured_mm:.1f} mm "
          f"(spread {spread_mm:.1f} mm over {len(readings)} readings)")
    for name, d in predicted.items():
        print(f"  {name:12s} predicts {d * 1e3:.1f} mm, error {d * 1e3 - measured_mm:+.1f} mm")
else:
    measured_mm = spread_mm = None
    print("physical.json has no balance readings: the predictions cannot be checked. "
          + ", ".join(f"{k} {v * 1e3:.1f} mm" for k, v in predicted.items()))

## Friction from a tilt

Put the object on a board and raise one end slowly. On a slope of angle $\theta$, gravity pulls the object along the slope with the force $m g \sin\theta$ and presses it into the board with $N = m g \cos\theta$. Static friction can hold it as long as the force along the slope is at most $\mu_s N$, so the object stays put while

$$
m g \sin\theta \le \mu_s \, m g \cos\theta \quad \Longleftrightarrow \quad \tan\theta \le \mu_s ,
$$

and it starts to slide at the angle where $\tan\theta = \mu_s$. The mass cancels, so the measurement needs only the angle. Each of your five readings gives one $\mu_s = \tan\theta$. You average the five $\mu_s$ values and report their standard deviation as the spread. The tangent is not linear, so averaging the angles first gives a slightly different number, and the difference grows with the spread of the readings.


In [ ]:
import math

import numpy as np


def friction_from_tilt(tilt_deg: list[float]) -> tuple[float, float]:
    """Mean and sample standard deviation of mu = tan(theta) over the tilt readings (degrees).

    Convert each reading to mu = tan(theta) first, then average the mu values; the spread is
    the standard deviation with ddof=1 (0.0 for a single reading).
    """
    raise NotImplementedError

In [ ]:
mu, mu_sd = friction_from_tilt(physical["tilt_deg"])
theta = np.degrees(np.arctan(mu))
print(f"static friction mu = {mu:.3f} (spread {mu_sd:.3f}) from {len(physical['tilt_deg'])} tilt readings; "
      f"the slip angle it implies is {theta:.1f} degrees")

## Pieces and centers of mass in the viewer

The viewer shows the decimated stage 4 mesh, half transparent, with the convex pieces in distinct colors under `world/collision` and the same pieces under `world/density`, red where the metallic map called them metal and blue elsewhere. Turn one group off to see the other. The long axis is the black line with its $-$ and $+$ ends labeled. The black point is the center of mass at uniform density, the red point (if you gave two materials) the center of mass with two densities, and the green point the measured balance point on the axis.


In [ ]:
def log_stage5(mesh_uv, pieces, metal, axis_info, coms, balance_point):
    rr.log("world/mesh", rr.Mesh3D(vertex_positions=mesh_uv["vertices"], triangle_indices=mesh_uv["faces"],
                                   albedo_factor=[0.8, 0.8, 0.8, 0.35]), static=True)
    palette = (np.array([[31, 119, 180], [255, 127, 14], [44, 160, 44], [214, 39, 40], [148, 103, 189],
                         [140, 86, 75], [227, 119, 194], [127, 127, 127], [188, 189, 34], [23, 190, 207]]))
    for i, p in enumerate(pieces):
        rr.log(f"world/collision/piece_{i:02d}", rr.Mesh3D(vertex_positions=p.vertices, triangle_indices=p.faces,
                                                           albedo_factor=palette[i % len(palette)]), static=True)
        density_color = [200, 60, 40] if metal[i] else [90, 140, 200]
        rr.log(f"world/density/piece_{i:02d}", rr.Mesh3D(vertex_positions=p.vertices, triangle_indices=p.faces,
                                                         albedo_factor=density_color), static=True)
    ends = [axis_info["center"] + axis_info["lo"] * axis_info["axis"],
            axis_info["center"] + axis_info["hi"] * axis_info["axis"]]
    rr.log("world/long_axis", rr.LineStrips3D([ends], colors=[[0, 0, 0]], radii=0.0005), static=True)
    rr.log("world/long_axis/ends", rr.Points3D(ends, labels=["- end", "+ end"], radii=0.002,
                                               colors=[[0, 0, 0], [0, 0, 0]]), static=True)
    rr.log("world/com", rr.Points3D(list(coms.values()), labels=[f"center of mass, {k}" for k in coms],
                                    radii=0.003, colors=[[0, 0, 0], [220, 0, 0]][:len(coms)]), static=True)
    if balance_point is not None:
        rr.log("world/balance", rr.Points3D([balance_point], labels=["measured balance point"], radii=0.003,
                                            colors=[[0, 160, 0]]), static=True)


coms = {"uniform": com_uniform} | ({"two densities": com_two} if com_two is not None else {})
balance_point = None
if measured_mm is not None:
    end = axis_info["hi"] if END_SIGN > 0 else axis_info["lo"]
    balance_point = axis_info["center"] + (end - END_SIGN * measured_mm / 1e3) * axis_info["axis"]
log_stage5(mesh_uv, pieces, metal, axis_info, coms, balance_point)

In [ ]:
show_rerun()

## Check

The union of the convex pieces must be within 10% of the mesh volume. A larger gap means that the pieces fill concave regions the object does not, and the simulated object would rest and roll on surfaces that are not there. Every inertia tensor must be symmetric and positive definite, and its principal moments must satisfy the triangle inequality $I_1 + I_2 \ge I_3$, which holds for every real body. Your two-density function, given a single material of 1 kg/m³, must reproduce the uniform model: the same center of mass within one and a half grid cells and the same inertia within 2%. A density that far from the real one also checks that the function scales the total to the measured mass.

The three measured dimensions are compared, smallest with smallest, with the mesh's oriented bounding box. This is the one place where a ruler checks the scale that stage 1 took from the board. The box leaves out the outermost 0.1% of the surface area on each side, because a single bump of the reconstruction would otherwise set its size, and the whole box is printed too. On the reference object a difference above 3% fails. On your object it prints a warning, since a ruler held against a real object is rougher than a scan.

The balance errors of both models are printed with no pass threshold, and the model whose prediction is closer to your readings is the one stage 6 exports. If your readings lie closer to the prediction measured from the other end, the cell warns that `END_SIGN` is probably wrong. On the reference object the values in `physical.json` are assumed, not measured, and the cell says so.


In [ ]:
import json
from pathlib import Path


def write_stage5(out: Path, pieces, record: dict) -> None:
    """Write stage5/ in one step: collision/piece_NN.obj for every convex piece, and physics.json."""
    with atomic_dir(out) as d:
        (d / "collision").mkdir()
        for i, p in enumerate(pieces):
            lines = [f"v {x:.6f} {y:.6f} {z:.6f}" for x, y, z in p.vertices]
            lines += [f"f {a} {b} {c}" for a, b, c in p.faces + 1]
            (d / "collision" / f"piece_{i:02d}.obj").write_text("\n".join(lines) + "\n")
        (d / "physics.json").write_text(json.dumps(record, indent=2) + "\n")

In [ ]:
import numpy as np


def robust_extents(mesh, to_obb: np.ndarray, trim: float = 0.001, n: int = 400_000, seed: int = 0) -> np.ndarray:
    """Extents (m) along the oriented box axes, leaving out the outermost `trim` share of the
    surface area on each side, so that a single bump of the reconstruction does not set the size."""
    import trimesh

    points, _ = trimesh.sample.sample_surface(mesh, n, seed=seed)
    q = points @ to_obb[:3, :3].T + to_obb[:3, 3]
    return np.percentile(q, 100 * (1 - trim), axis=0) - np.percentile(q, 100 * trim, axis=0)


def stage5_problems(mesh_volume: float, pieces_union: float, inertias: dict, extents_m, dims_mm: dict,
                    reference: bool, one_material=None, uniform=None, cell_m: float = 0.0, balance=None,
                    mu: float | None = None,
                    volume_tol: float = 0.10, dims_tol: float = 0.03) -> tuple[list[str], list[str]]:
    """(failures, warnings) of the stage 5 check.

    inertias maps a model name to its 3 x 3 tensor. dims are compared sorted, smallest with
    smallest; a mismatch fails on the reference and warns on your own object. one_material and
    uniform are (com, inertia) pairs: mass_properties_two_density run with one material must
    reproduce the uniform model, its center within 1.5 grid cells (cell_m, the cell side in m) and
    its inertia within 2%. balance is (predicted_mm, measured_mm, length_mm) for the named end.
    """
    failures, warnings = [], []
    if abs(pieces_union / mesh_volume - 1) > volume_tol:
        failures.append(f"the convex pieces' union volume is {pieces_union / mesh_volume - 1:+.1%} off the mesh "
                        f"volume (limit {volume_tol:.0%}): lower the CoACD threshold or raise the piece cap")
    for name, inertia in inertias.items():
        inertia = np.asarray(inertia, dtype=float)
        if inertia.shape != (3, 3) or not np.all(np.isfinite(inertia)):
            failures.append(f"{name}: the inertia is not a finite 3 x 3 matrix")
        elif not np.allclose(inertia, inertia.T, rtol=0, atol=1e-9 * np.abs(inertia).max()):
            failures.append(f"{name}: the inertia tensor is not symmetric")
        elif np.linalg.eigvalsh(inertia).min() <= 0:
            failures.append(f"{name}: the inertia tensor is not positive definite: is the mesh closed and "
                            "outward-facing?")
        else:
            p = np.sort(np.linalg.eigvalsh(inertia))
            if p[0] + p[1] < p[2] * (1 - 1e-6):
                failures.append(f"{name}: the principal moments break the triangle inequality, which no real body does")
    measured = np.sort(np.asarray(list(dims_mm.values()), dtype=float))
    model = np.sort(np.asarray(extents_m, dtype=float) * 1e3)
    if len(measured) != 3:
        failures.append(f"physical.json dims_mm needs three dimensions, it has {len(measured)}")
    else:
        off = model / measured - 1
        if np.abs(off).max() > dims_tol:
            text = (f"the mesh's box {np.round(model, 1)} mm is {np.round(off * 100, 1)}% off the measured "
                    f"{np.round(measured, 1)} mm (limit {dims_tol:.0%}): check the stage 1 scale and the ruler")
            (failures if reference else warnings).append(text)
    if one_material is not None:
        (c1, i1), (cu, iu) = one_material, uniform
        shift_mm = np.linalg.norm(np.asarray(c1) - np.asarray(cu)) * 1e3
        rel = np.linalg.norm(np.asarray(i1) - np.asarray(iu)) / np.linalg.norm(iu)
        limit_mm = 1.5 * cell_m * 1e3
        if not shift_mm < limit_mm or not rel < 0.02:
            failures.append(f"mass_properties_two_density with one material is {shift_mm:.2f} mm and {rel:.1%} off the "
                            f"uniform model (limits {limit_mm:.2f} mm, 2%): check the point masses, the inertia sum "
                            "and the scaling to mass_kg")
    if balance is not None:
        predicted, measured_mm, length = balance
        if abs(length - predicted - measured_mm) < abs(predicted - measured_mm):
            warnings.append(f"the measured balance point ({measured_mm:.1f} mm) is closer to the prediction from the "
                            f"other end ({length - predicted:.1f} mm) than from the named one ({predicted:.1f} mm): "
                            "check END_SIGN")
    if mu is not None and not 0.05 < mu < 1.5:
        warnings.append(f"mu = {mu:.2f} is unusual for everyday materials: check the tilt readings")
    return failures, warnings

In [ ]:
def test_friction_from_tilt_matches_tangent():
    mu, sd = friction_from_tilt([30.0] * 5)
    assert abs(mu - math.tan(math.radians(30.0))) < 1e-9 and sd == 0.0


test_friction_from_tilt_matches_tangent()
dims_extents = robust_extents(mesh, to_obb)
# your two-density function, run with one material, must give back the uniform model (scaled to the same mass)
one_material = mass_properties_two_density(mesh, pieces, np.zeros(len(pieces)), 1.0, 1.0, mass)
inertias = {"uniform": I_uniform} | ({"two densities": I_two} if I_two is not None else {})
length_mm = (axis_info["hi"] - axis_info["lo"]) * 1e3
failures, warnings = stage5_problems(
    mesh_volume, pieces_union, inertias, dims_extents, physical["dims_mm"], REFERENCE, one_material,
    (com_uniform, I_uniform), cell_mm / 1e3, None if measured_mm is None else (predicted["uniform"] * 1e3, measured_mm, length_mm), mu)
print(f"union of {len(pieces)} convex pieces {pieces_union * 1e6:.1f} mL against the mesh {mesh_volume * 1e6:.1f} mL "
      f"({pieces_union / mesh_volume - 1:+.1%}, limit 10%)")
print(f"inertia principal moments {np.round(np.linalg.eigvalsh(I_uniform) * 1e6, 2)} x 1e-6 kg m^2 (uniform)"
      + (f", {np.round(np.linalg.eigvalsh(I_two) * 1e6, 2)} (two densities)" if I_two is not None else ""))
print(f"mesh box {np.round(np.sort(dims_extents) * 1e3, 1)} mm (outermost 0.1% of the surface left out; whole box "
      f"{np.round(np.sort(extents) * 1e3, 1)} mm) against measured {sorted(physical['dims_mm'].values())} mm, limit 3%"
      + ("" if REFERENCE else " (a warning on your own object)"))
print(f"two-density function with one material: center of mass "
      f"{np.linalg.norm(one_material[0] - com_uniform) * 1e3:.2f} mm and inertia "
      f"{np.linalg.norm(one_material[1] - I_uniform) / np.linalg.norm(I_uniform):.2%} from the uniform model "
      f"(limits {1.5 * cell_mm:.2f} mm, 2%)")
closer = "uniform"
if measured_mm is not None:
    errors = {k: v * 1e3 - measured_mm for k, v in predicted.items()}
    closer = min(errors, key=lambda k: abs(errors[k]))
    print("balance point error (no threshold): " + ", ".join(f"{k} {e:+.1f} mm" for k, e in errors.items())
          + f"; stage 6 exports the {closer.replace('_', '-')} model")
if physical.get("assumed"):
    print("physical.json holds assumed values, not measurements: mass, friction, dimensions and balance point "
          "are stated assumptions for this object")
print(f"stage time {(time.time() - T0) / 60:.1f} min")
for w in warnings:
    print("warning:", w)
assert not failures, "; ".join(failures)  # nothing is written when a gate fails
models = {"uniform": {"com_m": com_uniform.tolist(), "inertia_kgm2": I_uniform.tolist(),
                      "balance_mm": predicted["uniform"] * 1e3}}
if com_two is not None:
    models["two_density"] = {"com_m": com_two.tolist(), "inertia_kgm2": I_two.tolist(),
                             "balance_mm": predicted["two_density"] * 1e3,
                             "density_kg_m3": {"metal": rho_metal, "other": rho_other},
                             "metal_pieces": np.nonzero(metal)[0].tolist(),
                             "metal_volume_fraction": float(volumes[metal_cells].sum() / volumes.sum())}
record = {"mass_kg": mass, "com_m": models[closer]["com_m"], "inertia_kgm2": models[closer]["inertia_kgm2"],
          "mu_static": mu, "mu_spread": mu_sd, "tilt_deg": physical["tilt_deg"], "model": closer, "models": models,
          "balance": None if measured_mm is None else {"end": balance["end"], "end_sign": END_SIGN,
                                                      "measured_mm": measured_mm, "spread_mm": spread_mm},
          "volume_m3": mesh_volume, "pieces_union_m3": pieces_union, "piece_metallic": metal_of_piece.tolist(),
          "assumed": bool(physical.get("assumed", False))}
write_stage5(PROJECT / "stage5", pieces, record)
if REFERENCE:
    (PROJECT / "stage5" / ".reference").touch()  # stage 6 then knows it runs on the reference
print("passed")

### Question 1. Convex pieces

Explain why a simulator collides convex pieces rather than the reconstructed mesh itself: what a contact test between two convex shapes can rely on that a test between two triangle meshes cannot, and what it costs at every simulation step. Then explain what CoACD's concavity threshold and its cap on the number of pieces trade against each other, and what goes wrong in contact if the union of the pieces is much larger than the object. Point to the volumes your decomposition cell printed and say which of them the simulator will treat as solid.

*Write your answer here.*

### Question 2. Friction from the slip angle

Derive $\mu_s = \tan\theta$ from the forces on an object resting on a slope of angle $\theta$, and state what the derivation assumes about the object and the board (consider whether it slides or tips first, and whether the slip starts at once over the whole contact). Averaging five values of $\tan\theta$ is not the same as taking the tangent of the mean angle: say which is larger and why. Point to the $\mu_s$ and spread your friction cell printed.

*Write your answer here.*

### Question 3. One density or two

State what the uniform-density model assumes about the inside of the object, and explain, with the center-of-mass sum and the parallel axis theorem, how giving part of the pieces a second, larger density moves the center of mass and changes the inertia tensor. Why does scaling both densities by one factor to the measured mass leave the center of mass where it was? The balance point measures only one coordinate of the center of mass: say which errors it can catch and which it cannot. Point to the balance errors your check printed and the model stage 6 will export.

*Write your answer here.*